# Investigation of RAB29 p.R175K variant carriers in the NBA

- Project: RAB GTPase gene investigation and burden analysis across populations with Parkinson’s disease
- Version: Python/3.10.19
- Last Updated: 13-OCT-2026

# Notebook Overview
- 1 RAB29 - p.R175K variant carriers in GP2 NBA R12 data
  - 1.1 Loading Python libraries
  - 1.2 Set Paths
  - 1.3 Defining functions
  - 1.4 Check RAB29 p.R175K variant carriers in NBA data

# *RAB29* - p.R175K variant carriers in GP2 NBA R12 data

## Loading Python libraries

In [ ]:
# Use the os package to interact with the environment
import os

# Bring in Pandas for Dataframe functionality
import pandas as pd

# Numpy for basics
import numpy as np

# Use StringIO for working with file contents
from io import StringIO

# Enable IPython to display matplotlib graphs
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors

# Import the iPython HTML rendering for displaying links to Google Cloud Console
from IPython.display import display, HTML

# Import urllib modules for building URLs to Google Cloud Console
import urllib.parse

# BigQuery for querying data
from google.cloud import bigquery

# Use pathlib for file path manipulation
import pathlib
from pathlib import Path

# Import Sys
import sys as sys

# Import Seaborn
import seaborn as sns

# Install plotnine: a ggplot2-compatible Python plotting package
from plotnine import *

# Import Glob
import glob

# Import Re
import re

# Import Subprocess
import subprocess

# Import CYVCF2
#from cyvcf2 import VCF

# Import tools for meta-analysis
from scipy import stats
from scipy.stats import fisher_exact

# Import collections
from collections import defaultdict, deque

# Import Tempfile
import tempfile

# Import Shutil
import shutil

# Always show all columns in a Pandas DataFrame
pd.set_option('display.max_columns', None)
pd.set_option('future.no_silent_downcasting', True)

## Set paths

In [ ]:
## GP2 v12.0
## Explicitly define release v11.0 path 
REL12_PATH = pathlib.Path(pathlib.Path.home(), 'workspace/gp2_tier2_eu_release12')

WGS_DATA_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/plink')
WGS_RELATED_PATH = pathlib.Path(REL12_PATH, 'wgs/dragen_joint_calling/related_samples')
CLINICAL_DATA_PATH = pathlib.Path(REL12_PATH, 'clinical_data/master_key_release12_final_vwb.csv')

## Defining functions

In [ ]:
# Utility routine for printing a shell command before executing it
def shell_do(command):
    print(f'Executing: {command}', file=sys.stderr)
    !$command
    
def shell_return(command):
    print(f'Executing: {command}', file=sys.stderr)
    output = !$command
    return '\n'.join(output)

# Utility routine for printing a query before executing it
def bq_query(query):
    print(f'Executing: {query}', file=sys.stderr)
    return pd.read_gbq(query, project_id=BILLING_PROJECT_ID, dialect='standard')

# Utility routine for display a message and a link
def display_html_link(description, link_text, url):
    html = f'''
    <p>
    </p>
    <p>
    {description}
    <a target=_blank href="{url}">{link_text}</a>.
    </p>
    '''

    display(HTML(html))


## Check *RAB29* p.R175K variant carriers in NBA data

In [ ]:
# Basic QC before extracting
NBA_DATA_PATH = f'/home/jupyter/workspace/gp2_tier2_eu_release12/imputed_genotypes/EUR'

!mkdir /home/jupyter/results/NBA/
WORK_DIR = f'/home/jupyter/results/NBA'
    
! /home/jupyter/tools/plink/plink2 \
--pfile {NBA_DATA_PATH}/chr1_EUR_release12_vwb  \
--geno 0.05 \
--make-pgen \
--max-alleles 2 \
--out {WORK_DIR}/chr1.qc

In [ ]:
# Extract RAB29 region using plink
! /home/jupyter/tools/plink/plink2 \
--pfile {WORK_DIR}/chr1.qc \
--chr 1 \
--from-bp 205767986 \
--to-bp 205775482 \
--make-bed \
--out {WORK_DIR}/RAB29

!rm {WORK_DIR}/chr1.qc*

In [ ]:
# Check RAB29 p.R175K variant
! /home/jupyter/tools/plink/plink1.9 \
--bfile {WORK_DIR}/RAB29 \
--snp chr1:205770430:C:T \
--keep-allele-order \
--recode A \
--out {WORK_DIR}/RAB29.R175K

In [ ]:
# Find IDs for all RAB29 variant carriers

input_file = os.path.join(WORK_DIR, 'RAB29.R175K.raw')

filename = os.path.basename(input_file)
print('File:', filename)

# Extract the variant name
parts = filename.split('.')
if len(parts) < 3:
    raise ValueError('Unexpected filename format')

variant_name = parts[2]
list_var_name = variant_name + '_list'

# Read the file (space-separated)
df = pd.read_csv(input_file, sep=r'\s+')
df.columns = df.columns.str.strip()

# Make sure we have at least 7 columns
if df.shape[1] < 7:
    raise ValueError('File has fewer than 7 columns')

# 7th column (index 6)
genotype_col = df.columns[6]

# Ensure numeric
df[genotype_col] = pd.to_numeric(df[genotype_col], errors='coerce')

# Carriers: genotype 1 or 2
carriers = df[df[genotype_col].isin([1, 2])]['IID']
carrier_list = carriers.astype(str).tolist()

# Store in dictionary (optional, if you still want this structure)
variant_carriers = {variant_name: carriers}

# Output
if len(carrier_list) == 0:
    print('No carriers with value 1 or 2 in 7th column')
else:
    print('Stored in variable:', list_var_name)
    for iid in carrier_list:
        print('  ', iid)

In [ ]:
# IDs of variant carriers
R175K_list = carrier_list 

# Make sure both sides are the same type
clinical_data_df_pheno['GP2ID'] = clinical_data_df_pheno['GP2ID'].astype(str)

# Filter clinical data
filtered_clinical_data_R175K = clinical_data_df_pheno[
    clinical_data_df_pheno['GP2ID'].isin(R175K_list)
]

# Save the filtered data
output_path = '/home/jupyter/results/NBA/master_key_release12_final_vwb_RAB29_R175K_NBA.csv'
filtered_clinical_data_R175K.to_csv(output_path, index=False)

print(f"Saved {len(filtered_clinical_data_R175K)} rows to:", output_path)